# 08-多轮对话 - Kimi API

本文档演示 Kimi API 的多轮对话功能。

In [1]:
from openai import OpenAI
import os
import json
from typing import List, Dict, Optional
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 基础多轮对话

In [2]:
# 基础多轮对话
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat(message: str) -> str:
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=messages,
    )
    
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    
    return reply

# 对话示例
print(f"User: 你好！")
print(f"Assistant: {chat('你好！')}")

print(f"\nUser: 我叫张三")
print(f"Assistant: {chat('我叫张三')}")

print(f"\nUser: 我叫什么名字？")
print(f"Assistant: {chat('我叫什么名字？')}")

User: 你好！
Assistant: 你好！很高兴见到你，有什么我可以帮忙的吗？

User: 我叫张三
Assistant: 张三，你好！很高兴认识你。  
以后我就叫你“张三”啦，有什么想聊的或需要帮忙的，尽管说～

User: 我叫什么名字？
Assistant: 你刚才告诉我你叫张三，没错吧？


## 对话管理类

In [3]:
class KimiChat:
    """Kimi 对话管理类"""
    
    def __init__(
        self,
        model: str = "kimi-k2-turbo-preview",
        system_prompt: str = "You are a helpful assistant.",
        max_history: int = 10
    ):
        self.client = client
        self.model = model
        self.max_history = max_history
        self.messages: List[Dict[str, str]] = []
        
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def chat(self, message: str, stream: bool = False, **kwargs) -> str:
        self.messages.append({"role": "user", "content": message})
        self._manage_history()
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            stream=stream,
            **kwargs
        )
        
        if stream:
            reply_parts = []
            for chunk in response:
                content = chunk.choices[0].delta.content
                if content:
                    reply_parts.append(content)
                    print(content, end="", flush=True)
            reply = "".join(reply_parts)
            print()
        else:
            reply = response.choices[0].message.content
        
        self.messages.append({"role": "assistant", "content": reply})
        return reply
    
    def _manage_history(self):
        if len(self.messages) > self.max_history * 2 + 1:
            system = self.messages[0] if self.messages[0]["role"] == "system" else None
            self.messages = self.messages[-(self.max_history * 2):]
            if system:
                self.messages.insert(0, system)
    
    def clear_history(self, keep_system: bool = True):
        if keep_system and self.messages and self.messages[0]["role"] == "system":
            system = self.messages[0]
            self.messages = [system]
        else:
            self.messages = []
    
    def get_history(self) -> List[Dict[str, str]]:
        return self.messages.copy()

# 使用示例
chat = KimiChat(
    model="kimi-k2-turbo-preview",
    system_prompt="你是一个专业的编程助手。",
    max_history=5
)

print(f"Assistant: {chat.chat('Python 中如何实现单例模式？')}")
print(f"\nAssistant: {chat.chat('请给出具体的代码示例')}")
print(f"\nAssistant: {chat.chat('这个实现是线程安全的吗？')}")

print("\n历史记录:")
for i, msg in enumerate(chat.get_history(), 1):
    content = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
    print(f"{i}. {msg['role']}: {content}")

Assistant: 在 Python 里“单例（Singleton）”的核心诉求只有一个：**同一个类在进程生命周期内只能被实例化一次**。  
Python 没有 static 构造函数，也没有编译期宏，但语言足够动态，所以民间流传的实现方式非常多样。下面给出 6 种常见写法，按“推荐程度”从高到低排列，并注明适用场景与坑点，你可以直接抄，也可以按需裁剪。

------------------------------------------------
1. 模块级单例（最简单，官方推荐）
------------------------------------------------
Python 模块在 `sys.modules` 里只会被导入一次，天然单例。  
把“实例”伪装成模块的全局变量即可。

```python
# singleton.py
class _Logger:
    def __init__(self):
        self.counter = 0
    def log(self, msg):
        self.counter += 1
        print(f'[{self.counter}] {msg}')

# 对外暴露的唯一实例
logger = _Logger()
```

使用方：

```python
from singleton import logger   # 任何地方导入的都是同一个对象
logger.log('hello')
```

✅ 零依赖、线程安全、可序列化、可继承。  
❌ 无法阻止别人直接 ` _Logger()` 再创建一个。

------------------------------------------------
2. 元类控制 __call__（最正统，可继承）
------------------------------------------------
把“单例逻辑”下沉到元类，子类自动成为单例，且支持传参缓存。

```python
import threading

class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()

    def __call__

## 多轮对话 + 思考模式

In [4]:
# 多轮对话 + 思考模式
messages = [
    {"role": "system", "content": "You are a math tutor."}
]

def chat_with_thinking(message: str) -> tuple[str, Optional[str]]:
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(
        model="kimi-k2-thinking",
        messages=messages,
    )
    
    message_obj = response.choices[0].message
    content = message_obj.content
    reasoning = getattr(message_obj, 'reasoning_content', None)
    
    messages.append({"role": "assistant", "content": content})
    
    return content, reasoning

# 对话示例
print("Round 1:")
content, reasoning = chat_with_thinking("解方程 x² - 7x + 12 = 0")
print(f"User: 解方程 x² - 7x + 12 = 0")
if reasoning:
    print(f"🧠 思考: {reasoning[:100]}...")
print(f"📄 回复: {content}")

print("\nRound 2:")
content, reasoning = chat_with_thinking("再用求根公式验证一下")
print(f"User: 再用求根公式验证一下")
if reasoning:
    print(f"🧠 思考: {reasoning[:100]}...")
print(f"📄 回复: {content}")

Round 1:
User: 解方程 x² - 7x + 12 = 0
🧠 思考: 用户要求解一个二次方程：x² - 7x + 12 = 0

这是一个标准的二次方程，我可以使用以下几种方法来解：

**方法1：因式分解法**
这是最简单的方法。我需要找到两个数，它们的乘积是12（常...
📄 回复: 我来帮你解这个二次方程：x² - 7x + 12 = 0

### 方法一：因式分解法（最简便）

我们需要找到两个数，它们的**和是 -7**，**积是 12**。

这两个数是 **-3** 和 **-4**，因为：
- (-3) + (-4) = -7
- (-3) × (-4) = 12

所以方程可以写成：
$$(x - 3)(x - 4) = 0$$

根据零乘积性质，若两个因式乘积为0，则至少有一个因式为0：
$$x - 3 = 0 \quad \text{或} \quad x - 4 = 0$$

解得：
$$x = 3 \quad \text{或} \quad x = 4$$

---

### 方法二：求根公式（通用方法）

对于方程 ax² + bx + c = 0，公式为：
$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

这里 a = 1, b = -7, c = 12。

计算判别式：
$$\Delta = b^2 - 4ac = (-7)^2 - 4(1)(12) = 49 - 48 = 1$$

代入公式：
$$x = \frac{7 \pm \sqrt{1}}{2} = \frac{7 \pm 1}{2}$$

得到两个解：
$$x_1 = \frac{7 + 1}{2} = 4$$
$$x_2 = \frac{7 - 1}{2} = 3$$

---

### 验证答案

将 x = 3 代入原方程：
$$3^2 - 7 \times 3 + 12 = 9 - 21 + 12 = 0 \quad ✓$$

将 x = 4 代入原方程：
$$4^2 - 7 \times 4 + 12 = 16 - 28 + 12 = 0 \quad ✓$$

### 最终答案

方程的解为 **x = 3** 和 **x = 4**。

Round 2:
User: 再用求根公式验证一下
🧠 思考: 用户要求用求

## 多轮对话 + 工具调用

In [5]:
# 多轮对话 + 工具调用
def get_weather(city: str):
    weather_data = {
        "北京": {"weather": "晴朗", "temp": "25°C"},
        "上海": {"weather": "多云", "temp": "23°C"},
    }
    return weather_data.get(city, {"weather": "未知", "temp": "未知"})

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "获取天气信息",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
    }
]

messages = [
    {"role": "system", "content": "You are a helpful weather assistant."}
]

def chat_with_tools(user_message: str):
    messages.append({"role": "user", "content": user_message})
    
    while True:
        response = client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=messages,
            tools=tools,
        )
        
        choice = response.choices[0]
        message = choice.message
        
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [{
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in (message.tool_calls or [])] if message.tool_calls else None
        })
        
        if not message.tool_calls:
            return message.content
        
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 调用工具: {func_name}")
            
            if func_name == "get_weather":
                result = get_weather(**func_args)
            else:
                result = {"error": "Unknown tool"}
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": json.dumps(result)
            })

# 对话示例
print("User: 北京天气怎么样？")
reply = chat_with_tools("北京天气怎么样？")
print(f"Assistant: {reply}\n")

print("User: 上海呢？")
reply = chat_with_tools("上海呢？")
print(f"Assistant: {reply}\n")

print("User: 谢谢")
reply = chat_with_tools("谢谢")
print(f"Assistant: {reply}")

User: 北京天气怎么样？
🔧 调用工具: get_weather
Assistant: 北京今天的天气是晴朗，气温为25°C，是个非常适合外出活动的好天气！

User: 上海呢？
🔧 调用工具: get_weather
Assistant: 上海今天的天气是多云，气温为23°C，比北京稍微凉爽一些。

User: 谢谢
Assistant: 不客气！如果还有其他城市需要查询，随时告诉我。祝您有愉快的一天！
